In [99]:
import sys
import time
import requests
import pandas as pd
import numpy as np
from pathlib import Path
from bs4 import BeautifulSoup
print(sys.executable)
print(sys.version)
print("requests", requests.__version__)

c:\Developer\MediBot\.venv\Scripts\python.exe
3.14.6 (tags/v3.14.6:c63aec6, Jun 10 2026, 10:26:10) [MSC v.1944 64 bit (AMD64)]
requests 2.34.2


In [ ]:


DATA_DIR = Path("medical_data")
DATA_DIR.mkdir(exist_ok=True)

DB_DIR = "./medical_chroma_db"

EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

In [88]:
import xml.etree.ElementTree as ET

def fetch_medlineplus_data(query):
    url = "https://wsearch.nlm.nih.gov/ws/query"
    params = {"db": "healthTopics", "term": query}
    response = requests.get(url, params=params, timeout=30)
    response.raise_for_status()
    
    soup = BeautifulSoup(response.content,"xml")

    results = []

    for document in soup.find_all("document"):
        url = document.get("url","")
        fields = {}
        contents = document.find_all("content")

        for content in contents:
            name = content.get("name","")
            raw_text = content.get_text(" ", strip=True)
            clean_text = BeautifulSoup(raw_text, "html.parser").get_text(" ", strip=True)

            fields[name] = clean_text

        title = fields.get("title","")
        summary = fields.get("FullSummary","")

        if not summary:
            summary = fields.get("snippet","")

        if title and summary and url:
            results.append({
                "title": title,
                "content": summary,
                "url": url,
                "source": "MedlinePlus",
                "category": "general"
            })
    return results


query = "flu"
ans = fetch_medlineplus_data(query)

for txt in ans:
    print("Title: ",txt["title"])
    print("Summary: ",txt["content"][:300])
    print("Url: ",txt["url"])
    print("Source: ",txt["source"])

Title:  Flu
Summary:  What is the flu ? The flu , also called influenza , is a respiratory infection caused by viruses. Each year, millions of Americans get sick with the flu . Sometimes it causes mild illness. But it can also be serious or even deadly, especially for people over 65, newborn babies, and people with certa
Url:  https://medlineplus.gov/flu.html
Source:  MedlinePlus
Title:  H1N1 Flu (Swine Flu )
Summary:  Swine flu is an infection caused by a virus. It's named for a virus that pigs can get. People do not normally get swine flu , but human infections can and do happen.  In 2009 a strain of swine flu called H1N1 infected many people around the world. The virus is contagious and can spread from human to
Url:  https://medlineplus.gov/h1n1fluswineflu.html
Source:  MedlinePlus
Title:  Bird Flu
Summary:  What is bird flu ? Birds, just like people, can get the flu . Another name for bird flu is avian influenza . The viruses that cause bird flu normally only infect birds, includin

In [93]:
# Collecting topics

medical_records = []

for topic in TOPICS:

    try:
        records = fetch_medlineplus_data(topic)
        medical_records.extend(records)

        print(f"{topic}: {len(records)} records")

        time.sleep(1)
    except requests.RequestException as error:
        print(f"Error fetching: {topic}: {error}")

print(f"Total records: {len(medical_records)}")



    

diabetes: 10 records
hypertension: 10 records
asthma: 10 records
pneumonia: 10 records
anemia: 10 records
migraine: 10 records
depression: 10 records
heart disease: 10 records
kidney disease: 10 records
fever: 10 records
Total records: 100


In [98]:
# Removing duplicate topics

unique_records = {}

for unique in medical_records:
    unique_records[unique["url"]] = unique

medical_records = list(unique_records.values())

print(f"Unique records: {len(medical_records)}")

Unique records: 94


In [101]:
# Inspect the dataset

df = pd.DataFrame(medical_records)

if not df.empty:
    display(
        df[["title","source", "category", "url"]]
    )
else:
    print("No records found")

,title,source,category,url
0,Diabetes,MedlinePlus,general,https://medlineplus.gov/diabetes.html
1,Diabetes Type 1,MedlinePlus,general,https://medlineplus.gov/diabetestype1.html
2,Diabetes Type 2,MedlinePlus,general,https://medlineplus.gov/diabetestype2.html
3,Diabetes and Pregnancy,MedlinePlus,general,https://medlineplus.gov/diabetesandpregnancy.html
4,Diabetes in Children and Teens,MedlinePlus,general,https://medlineplus.gov/diabetesinchildrenandt...
...,...,...,...,...
89,Dengue,MedlinePlus,general,https://medlineplus.gov/dengue.html
90,Valley Fever,MedlinePlus,general,https://medlineplus.gov/valleyfever.html
91,Ebola,MedlinePlus,general,https://medlineplus.gov/ebola.html
92,Salmonella Infections,MedlinePlus,general,https://medlineplus.gov/salmonellainfections.html


In [109]:
from langchain_ollama import OllamaEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from datetime import datetime, timezone
from langchain_chroma import Chroma
import hashlib

In [ ]:
approved_urls = set()

approved_records = [
    record
    for record in medical_records
    if record["url"] in approved_urls
]

documents = []
for record in approved_records:
    doc = Document(
        page_content=record["content"],
        metadata = {
            "title" : record["title"],
            "source" : record["source"],
            "source_url" : record["url"],
            "retrieved_at": datetime.now(
                timezone.utc
            ).isoformat(),
            "licensed_review": True

        }
    )

    documents.append(doc)

print(f"Approved documents: {len(documents)}")


Approved documents: 0


In [104]:
# SPLIT THE DOCUMENTS INTO CHUNKS
text_splitters = RecursiveCharacterTextSplitter(
        chunk_size = 1000,
        chunk_overlap=200,
        separators=[
            "\n\n",
            "\n",
            ". ",
            " "
        ]
)

chunked_docs = text_splitters.split_documents(documents)
print(f"Total Chunks: {len(chunked_docs)}")


Total Chunks: 0


In [105]:
# Initialization of Embeddings
embeddings = OllamaEmbeddings(
     model="nomic-embed-text"
)

In [108]:
# Vector Store
vectorstore = Chroma(
    collection_name = "medical_data",
    embedding_function=embeddings,
    persist_directory=DB_DIR
)

In [111]:
# Inserting approved chunks
if chunked_docs:
    ids = [
        hashlib.sha256(
            (
                chunk.metadata["source_url"]
                + "|"
                + chunk.page_content
            ).encode("utf-8")
        ).hexdigest()
        for chunk in chunked_docs
    ]

    vectorstore.add_documents(
        documents=chunked_docs,
        ids=ids
    )

    print("Knowledge base created!")
else:
    print("No approved chunks available to store.")

No approved chunks available to store.


In [112]:
# Creating retriever
general_retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={
        "k": 5,
        "filter": {
            "category": "general"
        }
    }
)

In [114]:
# Testing Retrievel
question = "What are the common symptoms of diabetes?"

retrieved_docs = general_retriever.invoke(question)

for i, doc in enumerate(retrieved_docs, 1):
    print(f"\nDocument {i}")
    print("Title:", doc.metadata.get("title"))
    print("Source:", doc.metadata.get("source"))
    print("URL:", doc.metadata.get("source_url"))
    print("Content:", doc.page_content[:500])
    print("-" * 60)